# Fine-tuning Llama-3.2-3B with Unsloth (LoRA SFT)

A clean, end-to-end walkthrough of supervised fine-tuning (SFT) with
[Unsloth](https://github.com/unslothai/unsloth): load a 4-bit model, attach
LoRA adapters, prepare a chat dataset, train on assistant responses only, and
run inference.

The notebook stops at inference. Saving, GGUF/Ollama export, and deployment are
covered separately.

**Model:** `unsloth/Llama-3.2-3B-Instruct` &nbsp;•&nbsp; **Dataset:**
[FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) (ShareGPT)


## 1. Setup

Install Unsloth and pinned versions of `transformers` / `trl`.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Load Model and Tokenizer

`FastLanguageModel.from_pretrained` downloads the model, applies 4-bit
quantization, and picks the compute dtype automatically.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048   # Auto-supports RoPE scaling internally
dtype = None            # None = auto-detect (fp16 for T4/V100, bf16 for Ampere+)
load_in_4bit = True     # 4-bit quantization to reduce memory use

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 3. Configure LoRA

LoRA freezes the base weights and trains a small set of low-rank adapter
matrices, so we update roughly 1–10% of the parameters instead of all of them.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,            # 0 is optimized
    bias = "none",               # "none" is optimized
    use_gradient_checkpointing = "unsloth",  # less VRAM, supports longer context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 4. Load and Prepare Dataset

FineTome-100k is in ShareGPT style (`{"from", "value"}`). `standardize_sharegpt`
converts it to the generic HF chat format (`{"role", "content"}`).

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt

dataset = load_dataset("mlabonne/FineTome-100k", split = "train")
dataset = standardize_sharegpt(dataset)

## 5. Apply Chat Template

`get_chat_template` installs the `llama-3.1` conversation format on the
tokenizer. We then render each conversation into a single `text` string that the
trainer will tokenize.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False,
                                           add_generation_prompt = False)
             for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)

Inspect one raw conversation:

In [ ]:
dataset[5]["conversations"]

And the same conversation after the chat template is applied.

*Note:* the Llama 3.1 template prepends a `"Cutting Knowledge Date: December 2023\nToday Date: ..."`
system line by default — that is expected.

In [ ]:
dataset[5]["text"]

## 6. Configure SFT Training

We use TRL's `SFTTrainer`. We run 60 steps for a quick demo; for a full run set
`num_train_epochs = 1` and remove `max_steps`.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

## 7. Response-Only Training

`train_on_responses_only` masks the loss on everything except the assistant
turns, so the model learns to *produce* answers rather than to also predict the
user's prompts. Unsloth auto-detects the instruction and response markers from
the chat template.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

## 8. Inspect Training Data

Decode one tokenized example to see the full input:

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

Now decode the **labels** (masked positions shown as spaces). Only the assistant response should remain:

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x
                  for x in trainer.train_dataset[5]["labels"]])

The system and user prompts are masked — training loss is computed on the assistant response only.

## 9. Train

Record GPU memory before training:

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

## 10. Training / Memory Statistics

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## 11. Inference

Switch to inference mode and generate. We use `temperature = 1.5` with
`min_p = 0.1`.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")
FastLanguageModel.for_inference(model)   # native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,   # required for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64,
                         use_cache = True, temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)